###Fine-Tuning BERT to Perform Sentiment Analysis

In [ ]:
#Install necessary packages
!pip -q install transformers
!pip -q install datasets


In [ ]:
from huggingface_hub import login

# Authenticate with the Hugging Face Hub.
# This opens a prompt to enter a personal access token
# (https://huggingface.co/settings/tokens). Authentication isn't strictly
# required for public datasets/models, but it helps avoid rate-limiting
# issues and is required for private/gated repositories.
login()

In [ ]:
from datasets import load_dataset

# Load the IMDB movie review dataset (50,000 labeled reviews for binary
# sentiment classification) from the Hugging Face Hub.
# Note: "stanfordnlp/imdb" is used instead of the original "imdb" repo,
# since the unnamespaced version is no longer compatible with recent
# versions of huggingface_hub.
imdb = load_dataset("stanfordnlp/imdb")
imdb

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer that matches the pretrained DistilBERT model.
# Using the same tokenizer the model was originally trained with is
# essential, since it ensures the vocabulary and special tokens
# (e.g. [CLS], [SEP], [PAD]) align with what the model expects.
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
def tokenize(samples):
    # Tokenize the raw text into input IDs and attention masks.
    # truncation=True ensures sequences longer than the model's max
    # input length are cut off, preventing errors during training.
    # Note: padding is intentionally left out here — it's handled later
    # by the data collator, which pads dynamically per batch instead of
    # padding every example to a fixed global length (more efficient).
    return tokenizer(samples['text'], truncation=True)


# Apply the tokenizer across the entire dataset.
# batched=True processes multiple examples at once (instead of one at a
# time), which is significantly faster.
tokenized_imdb = imdb.map(tokenize, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

# Create a data collator that dynamically pads each batch to the length
# of its longest sequence (instead of padding the whole dataset to a fixed
# max length), which is more memory-efficient.
# return_tensors='np' is used because 'tf' is no longer supported directly
# by the collator in recent transformers versions; to_tf_dataset() converts
# the resulting NumPy arrays into TensorFlow tensors automatically.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='np')

# Convert the tokenized training split into a tf.data.Dataset.
# Shuffling is enabled to prevent the model from learning any unintended
# ordering in the training data.
train_data = tokenized_imdb['train'].to_tf_dataset(
    columns=['attention_mask', 'input_ids', 'label'],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator
)

# Convert the tokenized test split into a tf.data.Dataset for validation.
# Shuffling is disabled here since order doesn't matter (and keeping it
# consistent makes evaluation results easier to compare across epochs).
validation_data = tokenized_imdb['test'].to_tf_dataset(
    columns=['attention_mask', 'input_ids', 'label'],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator
)

In [ ]:
from tensorflow.keras.optimizers import Adam
from transformers import TFAutoModelForSequenceClassification

# Load a pretrained DistilBERT model and add a classification head on top.
# num_labels=2 configures the model for binary classification (e.g. positive/negative sentiment).
model = TFAutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# Compile the model with the Adam optimizer.
# A small learning rate (2e-5) is used because we are fine-tuning a pretrained model,
# not training from scratch — large updates could destroy the pretrained weights.
# Note: when no loss is explicitly provided, TFAutoModelForSequenceClassification
# automatically uses an appropriate loss (sparse categorical cross-entropy) internally.
model.compile(
    optimizer=Adam(learning_rate=2e-5),
    metrics=['accuracy']
)

# Train the model on the prepared TensorFlow datasets.
# validation_data is used to monitor performance on unseen data after each epoch,
# helping detect overfitting.
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=3
)